# Vector Retriever

You will create a vector retriever using the Neo4j GraphRAG Python package with AWS Bedrock.

You will be able to review how the vector index is used to retrieve similar results and how the context can be used by an LLM to provide a response.

**Prerequisites:** Complete [02 Embeddings](02_embeddings.ipynb) first to populate the graph with embeddings and create the vector index.

---

Import the required Python modules and set up the AWS Bedrock configuration.

## Install Dependencies

First, install the required packages. This only needs to be run once per session.

In [ ]:
# Install neo4j-graphrag with Bedrock support
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" python-dotenv pydantic-settings -q

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.generation import GraphRAG

from data_utils import Neo4jConnection, get_llm, get_embedder

## Connect to Neo4j

Create and verify the connection to your Neo4j graph database.

In [ ]:
neo4j = Neo4jConnection().verify()
driver = neo4j.driver

## Initialize LLM and Embedder

Set up the Large Language Model (LLM) and the embedding model you will use in graph retrieval-augmented generation (GraphRAG) workflows.

- **LLM**: Uses AWS Bedrock's Claude model via the `BedrockLLM` interface.
- **Embedder**: Uses AWS Bedrock's Amazon Titan Embeddings via the `BedrockEmbeddings` class.

In [ ]:
# Initialize LLM and Embedder from AWS Bedrock
llm = get_llm()
embedder = get_embedder()

print(f"LLM: {llm.model_id}")
print(f"Embedder: {embedder.model_id}")

## Initialize Vector Retriever

Set up the vector-based retriever for semantic search over your Neo4j knowledge graph.

> Vector search enables semantic retrieval of text chunks from your Neo4j graph.  
> Instead of keyword matching, it finds the most contextually similar passages to your query, even if the wording is different.

In [ ]:
# Initialize Vector Retriever
vector_retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text']
)

print("Vector Retriever initialized!")

The **VectorRetriever** class:
- Connects to the Neo4j database using the provided `driver`.
- Uses the `chunkEmbeddings` vector index for efficient semantic retrieval.
- The `embedder` generates embeddings for the query.
- Returns the `text` property from matching chunks.

> **Tip:**  
> You can modify the `return_properties` list to include additional properties from the retrieved nodes.

## Simple Vector Search Diagnostic 

You can use the vector retriever to search for semantically similar data.

Test the vector search by retrieving the top 5 most relevant text chunks from the Neo4j knowledge graph for the given query.

In [ ]:
# Simple Vector Search
query = "What products does Apple make?"
result = vector_retriever.search(query_text=query, top_k=5)

print(f"Query: \"{query}\"")
print(f"Number of results returned: {len(result.items)}\n")
for item in result.items:
    score = item.metadata.get('score', 'N/A')
    node_id = item.metadata.get('id', 'N/A')
    content_preview = str(item.content)[:100]
    print(f"Score: {score:.4f}, Content: {content_preview}..., id: {node_id}")

**How it works:**  
1. The example `query`, "What products does Apple make?", is created
2. `vector_retriever.search()` runs the query and returns the top 5 matches based on vector similarity.
3. The results are formatted displaying:
    * The similarity score (`Score`)
    * A snippet of the retrieved content (`Content`)
    * The unique identifier for each chunk (`id`)

This diagnostic helps you verify that the vector search is working and inspect the quality of the top results for your query.

> **Tip:**
> Inspecting the returned results to verify relevance can help you to adjust your chunking or embedding strategy.

## Graph Retrieval-Augmented Generation (GraphRAG) Query

You can use the `GraphRAG` class to create a graph retrieval-augmented generation (GraphRAG) pipeline.

The `GraphRAG` class combines a Large Language Model (LLM) with a vector-based retriever to answer questions using both semantic search and generative reasoning.

In [ ]:
# Initialize GraphRAG and Perform Search
query = "What products does Apple make?"
rag = GraphRAG(
    llm=llm,
    retriever=vector_retriever
)
response = rag.search(query, retriever_config={"top_k": 5}, return_context=True)

print(f"Query: \"{query}\"")
print(f"Number of results returned: {len(response.retriever_result.items)}\n")
print("Answer:")
print(response.answer)

**How it works:**  
1. The retriever (`vector_retriever`) finds the most relevant text chunks from the Neo4j graph based on the input query.
2. The LLM (`llm`) uses the retrieved context to generate a natural language answer.
3. The `rag` pipeline is used to `search`.
4. The `answer` in the `response` is printed

The `GraphRAG` pipeline provides context-aware, accurate answers grounded in your knowledge graph data.

## Try Different Queries

Experiment with the vector retriever by modifying the `query`.

In [ ]:
# Try different queries
queries = [
    "What services does Apple offer?",
    "When does Apple's fiscal year end?",
    "Tell me about Apple's product line"
]

for query in queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 60)
    response = rag.search(query, retriever_config={"top_k": 3})
    print(f"Answer: {response.answer}")

## Summary

In this notebook, you learned:

1. **VectorRetriever** - Uses vector similarity to find relevant chunks
2. **Semantic search** - Finds content by meaning, not just keywords
3. **GraphRAG** - Combines retrieval with LLM generation for intelligent answers

---

**Next:** [Vector Cypher Retriever](04_vector_cypher_retriever.ipynb)

In [ ]:
# Cleanup
neo4j.close()